In [1]:
import pandas as pd
import duckdb

df_device_alarm = pd.DataFrame({
    "alarm_id": [
        101, 102, 103, 104, 105,
        201, 202, 203, 204,
        301, 302, 303, 304, 305
    ],
    "device_id": [
        "R05", "R05", "R05", "R05", "R05",
        "R16", "R16", "R16", "R16",
        "R34", "R34", "R34", "R34", "R34"
    ],
    "alarm_time": [
        "2026-08-01 08:00:00",
        "2026-08-01 09:00:00",
        "2026-08-01 10:00:00",
        "2026-08-01 11:00:00",
        "2026-08-01 12:00:00",

        "2026-08-01 08:30:00",
        "2026-08-01 09:30:00",
        "2026-08-01 10:30:00",
        "2026-08-01 11:30:00",

        "2026-08-01 07:00:00",
        "2026-08-01 08:00:00",
        "2026-08-01 09:00:00",
        "2026-08-01 10:00:00",
        "2026-08-01 11:00:00"
    ],
    "alarm_level": [
        "ERROR", "WARNING", "ERROR", "ERROR", "WARNING",
        "WARNING", "ERROR", "WARNING", "ERROR",
        "ERROR", "ERROR", "WARNING", "ERROR", "ERROR"
    ]
})

df_device_alarm["alarm_time"] = pd.to_datetime(
    df_device_alarm["alarm_time"]
)

df_device_alarm

,alarm_id,device_id,alarm_time,alarm_level
0,101,R05,2026-08-01 08:00:00,ERROR
1,102,R05,2026-08-01 09:00:00,WARNING
2,103,R05,2026-08-01 10:00:00,ERROR
3,104,R05,2026-08-01 11:00:00,ERROR
4,105,R05,2026-08-01 12:00:00,WARNING
5,201,R16,2026-08-01 08:30:00,WARNING
6,202,R16,2026-08-01 09:30:00,ERROR
7,203,R16,2026-08-01 10:30:00,WARNING
8,204,R16,2026-08-01 11:30:00,ERROR
9,301,R34,2026-08-01 07:00:00,ERROR


# SQL Daily Review：设备 ERROR 累计进度

## 题目背景

设备运行过程中会产生不同等级的告警。

现在希望观察每台设备的 `ERROR` 告警是如何随着时间逐步累积的。

例如，某台设备全部记录中一共发生了 4 次 `ERROR`：

```text
第 1 次 ERROR → 累计完成 25%
第 2 次 ERROR → 累计完成 50%
第 3 次 ERROR → 累计完成 75%
第 4 次 ERROR → 累计完成 100%
```

这里的“进度”表示：

> 截至当前记录已经发生的 ERROR 数量，占该设备全部 ERROR 数量的比例。

---

## 题目要求

保留设备的**所有告警记录**，不能提前过滤掉 `WARNING`。

对于每条记录计算：

1. 截至当前记录的累计 `ERROR` 数量；
2. 当前设备全部记录中的 `ERROR` 总数量；
3. `ERROR` 累计进度。

### ERROR 判断规则

只有：

```text
alarm_level = 'ERROR'
```

才计为一次 ERROR。

`WARNING` 不计入 ERROR 数量，但这一行仍然需要保留在最终结果中。

---

## 记录顺序

每台设备内部按照：

1. `alarm_time` 升序；
2. `alarm_id` 升序。

确定累计顺序。

---

## 计算公式

```text
error_progress
=
cumulative_error_count / total_error_count
```

结果保留 4 位小数。

---

## 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `alarm_id` | 告警编号 |
| `alarm_time` | 告警时间 |
| `alarm_level` | 告警等级 |
| `cumulative_error_count` | 截至当前累计 ERROR 数 |
| `total_error_count` | 当前设备全部 ERROR 数 |
| `error_progress` | ERROR 累计进度 |

---

## 最终排序

按照：

1. `device_id` 升序；
2. `alarm_time` 升序；
3. `alarm_id` 升序。

---

## 解题要求

- 使用窗口函数；
- 使用条件累计 `SUM()` 计算 `cumulative_error_count`；
- 累计窗口显式指定：

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

- 使用另一个窗口计算 `total_error_count`；
- `total_error_count` 的窗口不能按照时间逐行累计；
- 使用 CTE 先计算两个指标；
- 在外层计算 `error_progress`；
- 不使用 `GROUP BY`；
- 不提前过滤 `WARNING`。

In [ ]:
query = """
WITH cumulative_error AS (
    SELECT 
        device_id,
        alarm_id,
        alarm_time,
        alarm_level,

        SUM(
            CASE
                WHEN alarm_level = 'ERROR' THEN 1
                ELSE 0
            END
        ) OVER (
            PARTITION BY device_id
            ORDER BY alarm_time, alarm_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )::INTEGER AS cumulative_error_count,

        COUNT(
            CASE
                WHEN alarm_level = 'ERROR' THEN 1
            END
        ) OVER (
            PARTITION BY device_id
        ) AS total_error_count

    FROM df_device_alarm
)

SELECT
    device_id,
    alarm_id,
    alarm_time,
    alarm_level,
    cumulative_error_count,
    total_error_count,
    ROUND(
        cumulative_error_count / total_error_count,
        4
    ) AS error_progress
FROM cumulative_error
ORDER BY
    device_id,
    alarm_time,
    alarm_id;
"""

df = duckdb.execute(query).fetchdf()
df

,device_id,alarm_id,alarm_time,alarm_level,cumulative_error_count,total_error_count,error_progress
0,R05,101,2026-08-01 08:00:00,ERROR,1,3,0.333333
1,R05,102,2026-08-01 09:00:00,WARNING,1,3,0.333333
2,R05,103,2026-08-01 10:00:00,ERROR,2,3,0.666667
3,R05,104,2026-08-01 11:00:00,ERROR,3,3,1.000000
4,R05,105,2026-08-01 12:00:00,WARNING,3,3,1.000000
5,R16,201,2026-08-01 08:30:00,WARNING,0,2,0.000000
6,R16,202,2026-08-01 09:30:00,ERROR,1,2,0.500000
7,R16,203,2026-08-01 10:30:00,WARNING,1,2,0.500000
8,R16,204,2026-08-01 11:30:00,ERROR,2,2,1.000000
9,R34,301,2026-08-01 07:00:00,ERROR,1,4,0.250000
